# The head-direction system is a ring attractor whose structure is internally maintained during sleep

**Dataset**: DANDI 000056, Peyrache et al. (2015), *Nature Neuroscience*,
"Internally organized mechanisms of the head direction sense". Extracellular
recordings from the anterodorsal thalamic nucleus (ADn) and postsubiculum
(PoSub) of mice, together with dual-LED head tracking and scored behavioral
states (Awake / REM / Non-REM).

**Question**. Head-direction (HD) cells each fire when the animal faces a
particular direction. During wakefulness their population activity forms a
single "bump" on a ring, organized by preferred direction: cells with similar
preferred directions co-fire, cells ~180 deg apart anti-correlate. Is this
one-dimensional ring structure merely driven by sensory input while the
animal moves, or is it internally generated by the network itself? If the HD
system is a continuous ring attractor, the same correlation structure must
persist during sleep, when the head is still and no directional sensory
input is present.

**Approach** (five sessions from five mice, one per mouse):
1. Reconstruct head direction from the two LEDs and identify HD cells with
   wake tuning curves (mean vector length > 0.3, random-time shuffle p < 0.05).
2. Show that pairwise spike-count correlations, sorted by preferred
   direction, have ring (circulant) structure during wakefulness.
3. Show that the same structure is preserved during REM and Non-REM sleep
   (correlation-of-correlations ~0.9, far outside a label-permutation null).
4. Decode a virtual head direction from sleep population activity with the
   wake tuning curves: a single localized bump persists and drifts coherently
   around the ring, several times more continuously than a time-bin shuffle.

Data are streamed from the DANDI Archive with `remfile` and a local disk
cache; nothing is downloaded in full. Runtime is ~20-40 min on first run
(faster with a warm /tmp/remfile_cache_hd).

## Setup

In [ ]:
import os
import numpy as np
import matplotlib

matplotlib.use("Agg")  # headless: save figures only, never plt.show()
import matplotlib.pyplot as plt
import h5py
import remfile
from pynwb import NWBHDF5IO
import pynapple as nap
from tqdm import tqdm

rng = np.random.default_rng(42)
FIG = "figures"
os.makedirs(FIG, exist_ok=True)

# Published version 0.250624.0430 of DANDI 000056. One session per mouse,
# chosen for small file size and good HD-cell yield.
ASSETS = {
    "Mouse17-130128": "4cc64fe0-7b1e-404c-8b86-fb5659292830",
    "Mouse20-130514": "748aa5de-c0de-4aa7-a7ef-2aad2f87a7eb",
    "Mouse24-131213": "ada02790-6eb6-48ee-902d-9ba017303586",
    "Mouse25-140123": "bdb30f7d-ba69-4d2e-8504-2efab69cd8d7",
    "Mouse28-140310": "656704ea-a4cd-40f4-8158-a6533ebf2eee",
}
DECODE_SESSION = "Mouse28-140310"  # 20 HD cells: best population for decoding

BIN_CORR = 0.1   # 100 ms bins for spike-count correlations
NBINS_TC = 60    # tuning-curve bins over [0, 2 pi)
MVL_FLOOR = 0.3  # effect-size floor for HD-cell calls
N_SHUF = 500     # random-time null iterations per unit


def load_session(session):
    """Stream an NWB file from DANDI with a local disk cache."""
    url = f"https://api.dandiarchive.org/api/assets/{ASSETS[session]}/download/"
    disk_cache = remfile.DiskCache("/tmp/remfile_cache_hd")
    rem_file = remfile.File(url, disk_cache=disk_cache)
    h5py_file = h5py.File(rem_file, "r")
    io = NWBHDF5IO(file=h5py_file)
    nwbfile = io.read()
    return nap.NWBFile(nwbfile)


def get_hd_signal(nwb):
    """Head direction from the red and blue LEDs.

    Tracking failures are sentinel -1 values (not NaN); mask them.
    HD = angle of (red - blue) LED vector, in [0, 2 pi).
    """
    red = np.asarray(nwb["SubjectPosition/RedLED"].values)
    blue = np.asarray(nwb["SubjectPosition/BlueLED"].values)
    t = nwb["SubjectPosition/RedLED"].t
    valid = (red[:, 0] > 0) & (red[:, 1] > 0) & (blue[:, 0] > 0) & (blue[:, 1] > 0)
    hd = np.arctan2(red[:, 1] - blue[:, 1], red[:, 0] - blue[:, 0]) % (2 * np.pi)
    hd[~valid] = np.nan
    return nap.Tsd(t=t, d=hd), valid


def get_sorted_group(nwb):
    """Units as a TsGroup with spike times sorted (one session has an unsorted unit)."""
    units = nwb["units"]
    return nap.TsGroup({k: nap.Ts(np.sort(units[k].t)) for k in units.keys()})


def split_states(nwb):
    states = nwb["states"]
    return (states[states["label"] == "Awake"],
            states[states["label"] == "REM"],
            states[states["label"] == "Non-REM"])

## Per-session processing: tuning curves, HD-cell selection, per-state correlations

HD cells are identified during wakefulness by the mean vector length (MVL)
of spike angles. Because wake is fragmented into short epochs with nearly
constant HD, a circular time-shift shuffle is not a valid null here; we
instead redraw each unit's spike count of angles uniformly from the valid
wake HD samples (random-time null), and additionally require MVL > 0.3 as an
effect-size floor. Pairwise correlations are Pearson correlations of 100 ms
spike counts, computed separately within Awake, REM, and Non-REM epochs.

In [ ]:
def process_session(session):
    print("=" * 70, f"\n{session}")
    nwb = load_session(session)
    wake, rem, nrem = split_states(nwb)
    hd_tsd, valid = get_hd_signal(nwb)
    group = get_sorted_group(nwb)
    keys = list(group.keys())

    # Wake tuning curves (counts / occupancy -> Hz)
    tc_da = nap.compute_tuning_curves(group, hd_tsd, bins=NBINS_TC,
                                      range=(0, 2 * np.pi), epochs=wake,
                                      return_counts=True)
    occupancy = tc_da.attrs["occupancy"] / tc_da.attrs["fs"]
    counts = np.asarray(tc_da)
    with np.errstate(invalid="ignore", divide="ignore"):
        tc_hz = counts / occupancy[None, :]
    tc_hz = np.where(occupancy[None, :] > 0, tc_hz, np.nan)
    bin_dim = [d for d in tc_da.dims if d != "unit"][0]
    centers = np.asarray(tc_da.coords[bin_dim])

    # HD-cell selection: MVL of spike angles vs random-time null
    wake_mask = wake.in_interval(hd_tsd) >= 0
    hd_valid_a = np.asarray(hd_tsd.values)[valid & wake_mask]
    mvl = np.full(len(group), np.nan)
    pval = np.full(len(group), np.nan)
    for i, k in enumerate(tqdm(keys, desc=f"HD stats {session}")):
        ang = group[k].restrict(wake).value_from(hd_tsd)
        ang = ang[~np.isnan(ang)]
        n = len(ang)
        if n < 100:
            continue
        mvl[i] = np.abs(np.exp(1j * ang).mean())
        n_cap = min(n, 100_000)
        null = np.empty(N_SHUF)
        for s in range(N_SHUF):
            idx = rng.integers(0, len(hd_valid_a), size=n_cap)
            null[s] = np.abs(np.exp(1j * hd_valid_a[idx]).mean())
        pval[i] = (np.sum(null >= mvl[i]) + 1) / (N_SHUF + 1)
    is_hd = (mvl > MVL_FLOOR) & (pval < 0.05)
    hd_idx = np.where(is_hd)[0]
    pref = np.full(len(group), np.nan)
    pref[hd_idx] = centers[np.nanargmax(tc_hz[hd_idx], axis=1)]
    print(f"HD cells: {len(hd_idx)} of {len(group)}")

    # Per-state pairwise correlations among HD cells
    hd_group = nap.TsGroup({keys[i]: group[keys[i]] for i in hd_idx})
    corrs = {}
    for lab, ep in [("wake", wake), ("rem", rem), ("nrem", nrem)]:
        c = hd_group.count(BIN_CORR, ep)
        with np.errstate(invalid="ignore"):
            corrs[lab] = np.corrcoef(np.asarray(c.values, dtype=float).T)
    return dict(tc_hz=tc_hz, centers=centers, mvl=mvl, pval=pval, is_hd=is_hd,
                pref=pref, hd_idx=hd_idx, corrs=corrs)


sessions = {}
for session in ASSETS:
    sessions[session] = process_session(session)

n_hd = {s: int(d["is_hd"].sum()) for s, d in sessions.items()}
print("\nHD cells per session:", n_hd, " total:", sum(n_hd.values()))

## Raw data: the activity bump tracks head direction during wakefulness

Spikes of the 20 HD cells of Mouse28-140310, sorted by preferred direction,
during 40 s of exploration in which the animal turns through the full
circle. The band of active cells sweeps through the sorted population as the
head turns: population activity is a bump on the ring of preferred
directions.

In [ ]:
session = DECODE_SESSION
nwb = load_session(session)
wake, rem, nrem = split_states(nwb)
hd_tsd, valid = get_hd_signal(nwb)
group = get_sorted_group(nwb)
keys = list(group.keys())
d28 = sessions[session]
hd_idx, pref = d28["hd_idx"], d28["pref"]
order = np.argsort(pref[hd_idx])

# Find a 40 s wake window with the largest true angular span (unwrap-safe)
hd_wake = hd_tsd.restrict(wake)
ts, vs = hd_wake.t, np.asarray(hd_wake.values)
span, best, best_range = 40.0, None, -1
for i in range(0, len(ts) - 1, 200):
    j = np.searchsorted(ts, ts[i] + span)
    if j >= len(ts):
        break
    seg = vs[i:j]
    seg = seg[~np.isnan(seg)]
    if len(seg) < 100:
        continue
    cov = np.percentile(np.unwrap(seg), 95) - np.percentile(np.unwrap(seg), 5)
    if cov > best_range:
        best_range, best = cov, (ts[i], ts[i] + span)
t0, t1 = best
print(f"window {t0:.1f}-{t1:.1f} s, angular span {np.degrees(best_range):.0f} deg")

ep = nap.IntervalSet(t0, t1)
fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True,
                         gridspec_kw=dict(height_ratios=[1, 2.2]))
seg = hd_tsd.restrict(ep)
axes[0].plot(seg.t - t0, np.degrees(seg.values), ".", ms=1.5, color="k")
axes[0].set_ylabel("Head direction (deg)")
axes[0].set_yticks([0, 180, 360])
axes[0].set_title(f"Wake exploration ({session})", fontsize=10)
sorted_units = [keys[i] for i in hd_idx[order]]
for row, k in enumerate(sorted_units):
    spk = group[k].restrict(ep)
    axes[1].plot(spk.t - t0, np.full(len(spk), row), "|", ms=4, color="C0")
axes[1].set_ylabel("HD cells (sorted by\npreferred direction)")
axes[1].set_xlabel("Time in window (s)")
axes[1].set_yticks([0, len(sorted_units) - 1])
axes[1].set_yticklabels(["1", str(len(sorted_units))])
axes[1].set_ylim(-1, len(sorted_units))
plt.tight_layout()
plt.savefig(f"{FIG}/fig_raw_raster_wake.png", dpi=150, bbox_inches="tight")
plt.close()

## Wake tuning curves tile the ring

In [ ]:
centers = d28["centers"]
tc_hz = d28["tc_hz"]
n = len(hd_idx)
ncol = 5
nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, subplot_kw=dict(projection="polar"),
                         figsize=(2.0 * ncol, 2.0 * nrow))
for ax, u in zip(axes.ravel(), hd_idx[order]):
    tc_u = tc_hz[u]
    ax.plot(np.append(centers, centers[0]), np.append(tc_u, tc_u[0]),
            color="C0", lw=1.5)
    ax.set_title(f"u{u}", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.suptitle(f"Wake head-direction tuning curves ({session}, {n} HD cells)")
plt.tight_layout()
plt.savefig(f"{FIG}/fig_tuning_polar_M28.png", dpi=150, bbox_inches="tight")
plt.close()

# Preferred directions across all sessions + MVL selection statistic
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
ax = axes[0]
rng_jit = np.random.default_rng(1)
for s, d in sessions.items():
    prefs = np.degrees(d["pref"][d["hd_idx"]])
    ax.scatter(prefs, rng_jit.normal(0, 0.02, len(prefs)), s=12, alpha=0.7,
               label=s.replace("Mouse", "M"))
ax.set_xlabel("Preferred direction (deg)")
ax.set_yticks([])
ax.set_title(f"Preferred directions tile the ring ({sum(n_hd.values())} HD cells)")
ax.legend(frameon=False, fontsize=7, markerscale=1.2)
ax = axes[1]
all_mvl = np.concatenate([d["mvl"] for d in sessions.values()])
all_hd = np.concatenate([d["is_hd"] for d in sessions.values()])
b = np.linspace(0, 1, 40)
ax.hist(all_mvl[~all_hd & ~np.isnan(all_mvl)], bins=b, color="0.6", label="non-HD units")
ax.hist(all_mvl[all_hd], bins=b, color="C0", label="HD cells")
ax.axvline(MVL_FLOOR, color="r", ls="--", lw=1, label=f"MVL threshold {MVL_FLOOR}")
ax.set_xlabel("Mean vector length (wake)")
ax.set_ylabel("Units")
ax.set_title("HD-cell selection")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(f"{FIG}/fig_hdcell_selection.png", dpi=150, bbox_inches="tight")
plt.close()

## The ring structure: correlation depends on angular distance on the ring

Pairwise correlations among HD cells, sorted by wake preferred direction.
The diagonal band (cells with similar preferred directions co-fire) and the
wrap-around corners (cells near 0 deg and 360 deg are neighbors on the ring)
are the signature of a one-dimensional circular manifold. The same pattern
appears in REM and Non-REM sleep, when there is no directional sensory
input. Gray rows/columns in Mouse28 mark one unit that is silent in sleep.

In [ ]:
cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad(color="0.85")
fig, axes = plt.subplots(len(sessions), 3, figsize=(9.5, 2.1 * len(sessions)),
                         constrained_layout=True)
for r, (s, d) in enumerate(sessions.items()):
    pref_s = d["pref"][d["hd_idx"]]
    o = np.argsort(pref_s)
    for c, (lab, title) in enumerate([("wake", "Wake"), ("rem", "REM"), ("nrem", "NREM")]):
        C = d["corrs"][lab][np.ix_(o, o)]
        ax = axes[r, c]
        im = ax.imshow(np.ma.masked_invalid(C), cmap=cmap, vmin=-0.5, vmax=0.5,
                       origin="lower")
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(title, fontsize=11)
        if c == 0:
            ax.set_ylabel(s.replace("Mouse", "M"), fontsize=9)
fig.suptitle("Pairwise correlation matrices, HD cells sorted by wake preferred direction")
fig.colorbar(im, ax=axes, shrink=0.6, label="Pearson r", location="right")
plt.savefig(f"{FIG}/fig_corrmatrices_sorted.png", dpi=150, bbox_inches="tight")
plt.close()

## The wake ring profile is preserved during sleep

Left: mean pairwise correlation as a function of the angular distance
between the two cells' preferred directions. Correlation falls with distance
and goes negative near 180 deg, as expected for a bump on a ring, and the
profile is nearly identical across Wake, REM, and Non-REM. Right: every
pair's sleep correlation against its wake correlation, pooled across
sessions; both sleep states lie on the identity line.

In [ ]:
def upper_pairs(C):
    iu = np.triu_indices(C.shape[0], k=1)
    return C[iu]


def ang_dist(a, b):
    return np.abs(np.angle(np.exp(1j * (a - b))))


bins = np.linspace(0, np.pi, 13)
bc = (bins[:-1] + bins[1:]) / 2
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ax = axes[0]
colors = {"wake": "k", "rem": "r", "nrem": "b"}
labels = {"wake": "Wake", "rem": "REM", "nrem": "NREM"}
for key in ["wake", "rem", "nrem"]:
    all_d, all_c = [], []
    for s, d in sessions.items():
        pref_s = d["pref"][d["hd_idx"]]
        iu = np.triu_indices(len(pref_s), k=1)
        dd = ang_dist(pref_s[iu[0]], pref_s[iu[1]])
        cc = upper_pairs(d["corrs"][key])
        m = ~np.isnan(cc)
        all_d.append(dd[m])
        all_c.append(cc[m])
    all_d = np.concatenate(all_d)
    all_c = np.concatenate(all_c)
    inds = np.digitize(all_d, bins) - 1
    means = [np.nanmean(all_c[inds == i]) for i in range(len(bc))]
    sems = [np.nanstd(all_c[inds == i]) / np.sqrt(np.sum(inds == i)) for i in range(len(bc))]
    ax.errorbar(np.degrees(bc), means, yerr=sems, color=colors[key],
                label=labels[key], capsize=2, lw=1.5)
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("Angular distance between preferred directions (deg)")
ax.set_ylabel("Mean pairwise correlation")
ax.set_title("Ring profile: correlation vs. tuning distance")
ax.legend(frameon=False)

ax = axes[1]
all_w = np.concatenate([upper_pairs(d["corrs"]["wake"]) for d in sessions.values()])
all_r = np.concatenate([upper_pairs(d["corrs"]["rem"]) for d in sessions.values()])
all_n = np.concatenate([upper_pairs(d["corrs"]["nrem"]) for d in sessions.values()])
m = ~np.isnan(all_w) & ~np.isnan(all_r) & ~np.isnan(all_n)
all_w, all_r, all_n = all_w[m], all_r[m], all_n[m]
r_rem = np.corrcoef(all_w, all_r)[0, 1]
r_nrem = np.corrcoef(all_w, all_n)[0, 1]
ax.scatter(all_w, all_r, s=6, c="r", alpha=0.4, label=f"REM (r={r_rem:.2f})")
ax.scatter(all_w, all_n, s=6, c="b", alpha=0.4, label=f"NREM (r={r_nrem:.2f})")
lim = [-0.6, 0.8]
ax.plot(lim, lim, "k--", lw=0.8)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_xlabel("Wake pairwise correlation")
ax.set_ylabel("Sleep pairwise correlation")
ax.set_title(f"Structure preserved in sleep ({len(all_w)} pairs, {len(sessions)} sessions)")
ax.legend(frameon=False, markerscale=2)
plt.tight_layout()
plt.savefig(f"{FIG}/fig_ring_profile_and_preservation.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"pooled corr-of-corr: wake-REM r={r_rem:.2f}, wake-NREM r={r_nrem:.2f}")

## Statistical test against a label-permutation null

For each session we correlate the upper triangle of the wake correlation
matrix with that of each sleep state ("correlation of correlations"). The
null permutes unit labels in the sleep matrix, destroying the correspondence
between wake tuning and sleep correlations while preserving each matrix's
marginal statistics.

In [ ]:
def corr_of_corr(d, k1, k2):
    a, b = upper_pairs(d["corrs"][k1]), upper_pairs(d["corrs"][k2])
    m = ~np.isnan(a) & ~np.isnan(b)
    return np.corrcoef(a[m], b[m])[0, 1]


N_PERM = 1000
null_r, null_n = [], []
for s, d in sessions.items():
    nunits = len(d["hd_idx"])
    for _ in range(N_PERM // len(sessions)):
        perm = rng.permutation(nunits)
        a = upper_pairs(d["corrs"]["wake"])
        b = upper_pairs(d["corrs"]["rem"][np.ix_(perm, perm)])
        c = upper_pairs(d["corrs"]["nrem"][np.ix_(perm, perm)])
        m = ~np.isnan(a) & ~np.isnan(b)
        null_r.append(np.corrcoef(a[m], b[m])[0, 1])
        m = ~np.isnan(a) & ~np.isnan(c)
        null_n.append(np.corrcoef(a[m], c[m])[0, 1])
null_r, null_n = np.array(null_r), np.array(null_n)
obs_r = [corr_of_corr(d, "wake", "rem") for d in sessions.values()]
obs_n = [corr_of_corr(d, "wake", "nrem") for d in sessions.values()]
print("per-session corr-of-corr wake-REM:", np.round(obs_r, 3))
print("per-session corr-of-corr wake-NREM:", np.round(obs_n, 3))
print(f"null REM: mean {null_r.mean():.3f}, 99th pct {np.percentile(null_r, 99):.3f}")
print(f"null NREM: mean {null_n.mean():.3f}, 99th pct {np.percentile(null_n, 99):.3f}")
p_rem = (np.sum(null_r >= np.mean(obs_r)) + 1) / (len(null_r) + 1)
p_nrem = (np.sum(null_n >= np.mean(obs_n)) + 1) / (len(null_n) + 1)
print(f"mean observed vs null: REM p={p_rem:.4f}, NREM p={p_nrem:.4f}")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
b = np.linspace(-0.3, 1.0, 60)
ax.hist(null_r, bins=b, color="r", alpha=0.45, density=True, label="REM null (label perm.)")
ax.hist(null_n, bins=b, color="b", alpha=0.45, density=True, label="NREM null (label perm.)")
for v, c, lab in [(np.mean(obs_r), "r", f"observed wake-REM = {np.mean(obs_r):.2f}"),
                  (np.mean(obs_n), "b", f"observed wake-NREM = {np.mean(obs_n):.2f}")]:
    ax.axvline(v, color=c, lw=2)
    ax.text(v, ax.get_ylim()[1] * 0.9, lab, rotation=90, va="top", ha="right",
            color=c, fontsize=9)
ax.set_xlabel("Correlation of pairwise-correlation structure (wake vs sleep)")
ax.set_ylabel("Density")
ax.set_title("Sleep preserves the wake ring structure")
plt.tight_layout()
plt.savefig(f"{FIG}/fig_corr_of_corr_null.png", dpi=150, bbox_inches="tight")
plt.close()

## Decoding a virtual head direction during sleep

If the ring structure is internally maintained, population activity during
sleep should still trace a single bump on the same ring, and the bump's
position (an internally generated, "virtual" head direction) should drift
coherently rather than jump randomly. We decode angle from 100 ms spike
counts with the wake tuning curves under a Poisson Bayesian decoder
(Zhang et al. 1998), cross-checked against `nap.decode_bayes`. The control
permutes the time bins within each sleep epoch: identical posterior
distributions, no temporal continuity.

In [ ]:
def decode_counts(counts, tc_hz, bin_s):
    """Poisson Bayesian decode. counts (T, U); tc_hz (U, B) -> P (T, B)."""
    lam = np.clip(tc_hz * bin_s, 1e-12, None)
    logL = counts @ np.log(lam) - lam.sum(axis=0)[None, :]
    logL -= logL.max(axis=1, keepdims=True)
    P = np.exp(logL)
    P /= P.sum(axis=1, keepdims=True)
    return P


def circ_stats(P, centers):
    z = (P * np.exp(1j * centers[None, :])).sum(axis=1)
    return np.abs(z), np.angle(z) % (2 * np.pi)


def counts_per_epoch(group, ep, bin_s, smooth_bins=None):
    """Count spikes per epoch; optional uniform smoothing within each epoch."""
    all_c, all_t, lens = [], [], []
    for i in range(len(ep)):
        seg = nap.IntervalSet(ep.start[i], ep.end[i])
        c = group.count(bin_s, seg)
        v = np.asarray(c.values, dtype=float)
        if smooth_bins and smooth_bins > 1:
            k = np.ones(smooth_bins) / smooth_bins
            v = np.apply_along_axis(lambda x: np.convolve(x, k, mode="same"), 0, v)
        all_c.append(v)
        all_t.append(c.t)
        lens.append(len(v))
    return np.vstack(all_c), np.concatenate(all_t), lens


hd_group = nap.TsGroup({keys[i]: group[keys[i]] for i in hd_idx})
tc_da = nap.compute_tuning_curves(hd_group, hd_tsd, bins=NBINS_TC,
                                  range=(0, 2 * np.pi), epochs=wake)
tc_hz_dec = np.asarray(tc_da)
BIN = 0.1
SW = 5  # 500 ms smoothing for display decode

dec = {}
for lab, ep in [("wake", wake), ("rem", rem), ("nrem", nrem)]:
    C, tt, lens = counts_per_epoch(hd_group, ep, BIN)
    P = decode_counts(C, tc_hz_dec, BIN)
    R, ang = circ_stats(P, centers)
    dt = np.diff(tt)
    speed = np.abs(np.angle(np.exp(1j * np.diff(ang)))) / dt
    speed[dt > 2 * BIN] = np.nan  # mask across-epoch gaps
    dec.update({f"{lab}_R": R, f"{lab}_ang": ang, f"{lab}_t": tt,
                f"{lab}_speed": speed, f"{lab}_P": P})
    print(f"{lab}: median R={np.median(R):.3f}  "
          f"median speed={np.degrees(np.nanmedian(speed)):.1f} deg/s")

    Cs, tts, _ = counts_per_epoch(hd_group, ep, BIN, smooth_bins=SW)
    Ps = decode_counts(Cs, tc_hz_dec, BIN)
    Rs, angs = circ_stats(Ps, centers)
    dec.update({f"{lab}_smooth_P": Ps, f"{lab}_smooth_ang": angs,
                f"{lab}_smooth_t": tts})

    if lab in ("rem", "nrem"):
        Csh = C.copy()
        row0 = 0
        for n_i in lens:
            seg_idx = np.arange(row0, row0 + n_i)
            Csh[seg_idx] = Csh[seg_idx][rng.permutation(n_i)]
            row0 += n_i
        Psh = decode_counts(Csh, tc_hz_dec, BIN)
        Rsh, angsh = circ_stats(Psh, centers)
        speedsh = np.abs(np.angle(np.exp(1j * np.diff(angsh)))) / dt
        speedsh[dt > 2 * BIN] = np.nan
        dec.update({f"{lab}_binshuf_speed": speedsh})
        print(f"{lab} bin-shuffle: median speed="
              f"{np.degrees(np.nanmedian(speedsh)):.1f} deg/s")

# Wake validation: decoded vs actual HD (circular-safe interpolation)
t_w = dec["wake_t"]
t_led = hd_tsd.t
hd_v = np.asarray(hd_tsd.values)
mvalid = ~np.isnan(hd_v)
cos_i = np.interp(t_w, t_led[mvalid], np.cos(hd_v[mvalid]))
sin_i = np.interp(t_w, t_led[mvalid], np.sin(hd_v[mvalid]))
actual = np.angle(cos_i + 1j * sin_i) % (2 * np.pi)
err = np.angle(np.exp(1j * (dec["wake_ang"] - actual)))
print(f"wake decode median |err| = {np.degrees(np.median(np.abs(err))):.1f} deg")

# Cross-check the manual decoder against nap.decode_bayes on a wake subset
ep_check = nap.IntervalSet(t_w[0], t_w[0] + 300)
dec_ref, _ = nap.decode_bayes(tc_da, hd_group, ep_check, BIN)
m = (t_w >= t_w[0]) & (t_w <= t_w[0] + 300)
n_overlap = min(len(dec_ref), m.sum())
diff = np.angle(np.exp(1j * (np.asarray(dec_ref.values)[:n_overlap]
                             - dec["wake_ang"][m][:n_overlap])))
print(f"manual vs decode_bayes: median |diff| = "
      f"{np.degrees(np.median(np.abs(diff))):.2f} deg over {n_overlap} bins")

### Wake validation figure

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
t0w = t_w[len(t_w) // 3]
m = (t_w >= t0w) & (t_w <= t0w + 60)
axes[0].plot(t_w[m] - t0w, np.degrees(actual[m]), ".", ms=2, color="0.5",
             label="actual (LED)")
axes[0].plot(t_w[m] - t0w, np.degrees(dec["wake_ang"][m]), ".", ms=2,
             color="C0", label="decoded")
axes[0].set_ylabel("Head direction (deg)")
axes[0].set_xlabel("Time in wake window (s)")
axes[0].set_yticks([0, 180, 360])
axes[0].legend(frameon=False, markerscale=3, loc="upper right")
axes[0].set_title("Wake: decoded vs actual head direction")
err_deg = np.degrees(err)
axes[1].hist(err_deg, bins=np.linspace(-180, 180, 72), color="C0", density=True)
axes[1].axvline(0, color="k", lw=0.8)
med = np.median(np.abs(err_deg))
axes[1].axvline(med, color="r", ls="--", lw=1)
axes[1].axvline(-med, color="r", ls="--", lw=1)
axes[1].set_title(f"Wake decode error (median |err| = {med:.0f} deg)")
axes[1].set_xlabel("Decoded - actual HD (deg)")
axes[1].set_ylabel("Density")
axes[1].set_xlim(-180, 180)
plt.tight_layout()
plt.savefig(f"{FIG}/fig_decode_wake_validation.png", dpi=150, bbox_inches="tight")
plt.close()

### The bump drifts around the ring during REM and Non-REM sleep

Posterior probability over head direction (color) and its circular mean
(red line) during the strongest-bump 60 s of the longest REM and Non-REM
episodes of the session. The animal is asleep with its head still; the
decoded direction is entirely internally generated.

In [ ]:
def longest_epoch(ep):
    lengths = ep.end - ep.start
    i = int(np.argmax(lengths))
    return float(ep.start[i]), float(ep.end[i])


def strongest_window(lab, ep_bounds, win=60, search_s=600):
    t_s = dec[f"{lab}_smooth_t"]
    P_s = dec[f"{lab}_smooth_P"]
    R_s = np.abs((P_s * np.exp(1j * centers[None, :])).sum(axis=1))
    best_t0, best_R = None, -1
    for cand in np.arange(ep_bounds[0], min(ep_bounds[1] - win,
                                            ep_bounds[0] + search_s), 5):
        m = (t_s >= cand) & (t_s < cand + win)
        if m.sum() < 100:
            continue
        if R_s[m].mean() > best_R:
            best_R, best_t0 = R_s[m].mean(), cand
    return best_t0, best_t0 + win


for lab, title, fname in [
    ("rem", "REM sleep: internally generated bump of HD activity drifts around the ring",
     "fig_decode_rem.png"),
    ("nrem", "NREM sleep: the HD bump is maintained and drifts without sensory input",
     "fig_decode_nrem.png"),
]:
    ep_b = longest_epoch(rem if lab == "rem" else nrem)
    t0d, t1d = strongest_window(lab, ep_b)
    t_s = dec[f"{lab}_smooth_t"]
    m = (t_s >= t0d) & (t_s <= t1d)
    Pn = dec[f"{lab}_smooth_P"][m]
    ang = dec[f"{lab}_smooth_ang"][m]
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.imshow(Pn.T, origin="lower", aspect="auto", cmap="viridis",
              extent=[t_s[m][0] - t0d, t_s[m][-1] - t0d, 0, 360],
              interpolation="nearest")
    ax.plot(t_s[m] - t0d, np.degrees(ang), color="r", lw=1.0, alpha=0.9)
    ax.set_yticks([0, 180, 360])
    ax.set_ylabel("HD (deg)")
    ax.set_xlabel(f"Time in {lab.upper()} episode (s)")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{FIG}/{fname}", dpi=150, bbox_inches="tight")
    plt.close()
    print("saved", fname)

### Drift is temporally continuous, and the bump stays localized

Left: angular speed of the decoded virtual HD (log scale). During REM the
bump drifts at about the same speed as the real head during wakefulness;
during Non-REM it drifts faster, as reported by Peyrache et al. (2015).
Shuffling time bins within the same sleep epochs makes the decoded angle
3-8x jumpier, so continuity is a property of the activity, not of the
decoder. Right: posterior concentration R is as high in sleep as in wake; a
single localized bump persists.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
ax = axes[0]
speeds = [dec["wake_speed"][~np.isnan(dec["wake_speed"])],
          dec["rem_speed"][~np.isnan(dec["rem_speed"])],
          dec["nrem_speed"][~np.isnan(dec["nrem_speed"])],
          dec["rem_binshuf_speed"][~np.isnan(dec["rem_binshuf_speed"])],
          dec["nrem_binshuf_speed"][~np.isnan(dec["nrem_binshuf_speed"])]]
names = ["Wake", "REM", "NREM", "REM\nbin-shuf", "NREM\nbin-shuf"]
cols = ["k", "r", "b", "0.65", "0.65"]
bp = ax.boxplot([np.degrees(s) for s in speeds], tick_labels=names,
                showfliers=False, patch_artist=True, medianprops=dict(color="k"))
for patch, c in zip(bp["boxes"], cols):
    patch.set_facecolor(c)
    patch.set_alpha(0.55)
ax.set_ylabel("Decoded angular speed (deg/s)")
ax.set_title("Bump drift is temporally continuous in sleep")
ax.set_yscale("log")

ax = axes[1]
bp = ax.boxplot([dec["wake_R"], dec["rem_R"], dec["nrem_R"]],
                tick_labels=["Wake", "REM", "NREM"], showfliers=False,
                patch_artist=True, medianprops=dict(color="k"))
for patch, c in zip(bp["boxes"], ["k", "r", "b"]):
    patch.set_facecolor(c)
    patch.set_alpha(0.55)
ax.set_ylabel("Posterior concentration R")
ax.set_title("A single localized bump persists in sleep")
plt.tight_layout()
plt.savefig(f"{FIG}/fig_decode_stats.png", dpi=150, bbox_inches="tight")
plt.close()

## Summary

Across five sessions from five mice (53 HD cells of 159 recorded units in
ADn/PoSub):

1. **Ring structure during wakefulness.** HD-cell pairwise correlations
   sorted by preferred direction show the circulant pattern expected of a
   bump on a ring: strong positive correlation at small angular distances,
   negative correlation near 180 deg, and wrap-around at the 0/360 deg
   boundary.
2. **Internal maintenance during sleep.** The same correlation structure is
   present in REM and Non-REM sleep (pooled correlation-of-correlations
   r ~ 0.9 for both states; per-session means ~0.88 and ~0.78, far outside
   the label-permutation null). Because the sleeping animal receives no
   directional sensory input, the ring must be maintained by the network
   itself, the defining property of a continuous ring attractor.
3. **A coherent internal bump.** Bayesian decoding with wake tuning curves
   recovers a single localized bump of activity during sleep that drifts
   continuously around the ring (REM drift ~90 deg/s, close to the wake
   head-speed of ~115 deg/s; Non-REM faster at ~270 deg/s; time-bin shuffles
   jump 3-8x faster). The HD representation is not just statistically
   preserved in sleep: it is an active, moving internal variable.

These results reproduce the central findings of Peyrache et al. (2015) from
the same data, streamed directly from the DANDI Archive.